# Legacy-control 5K raw vs polished PLY

This bounded A100 comparison reproduces the passing legacy-control 5K diagnostic once. It stages the verified lineage once, trains one deterministic arm, preserves the trainer's raw PLY byte-for-byte, runs the unchanged production PLY polish, and publishes both PLYs with the polish report. A rejected polish candidate is still kept for inspection. It never starts a full 120K run and never overwrites the learned, legacy, ablation, or floor-diagnostic result.

In [ ]:
INPUT_FOLDER = ""  # @param {type:"string"}
RUNTIME_PROFILE = "a100_legacy_control_5k"


In [ ]:
import json, os, re, shutil, subprocess
gpu_line = subprocess.run([
    'nvidia-smi', '--query-gpu=name,memory.total',
    '--format=csv,noheader,nounits',
], check=True, capture_output=True, text=True).stdout.splitlines()[0]
gpu_name, memory_mib = (part.strip() for part in gpu_line.rsplit(',', 1))
vram_gib = float(memory_mib) / 1024.0
is_a100 = re.search(r'\bA100\b', gpu_name, flags=re.IGNORECASE) is not None
assert is_a100 and vram_gib >= 75.0, (
    f'A100 with at least 75 GiB VRAM required; detected '
    f'{gpu_name} ({vram_gib:.1f} GiB)'
)
disk_gib = shutil.disk_usage('/content').free / (1024 ** 3)
assert disk_gib >= 80.0, f'At least 80 GiB local disk required; detected {disk_gib:.1f}'
host_ram_gib = os.sysconf('SC_PHYS_PAGES') * os.sysconf('SC_PAGE_SIZE') / (1024 ** 3)
assert host_ram_gib >= 100.0, f'At least 100 GiB host RAM required; detected {host_ram_gib:.1f}'
print(json.dumps({'runtime_profile': RUNTIME_PROFILE, 'gpu': gpu_name, 'vram_gib': round(vram_gib, 1), 'host_ram_gib': round(host_ram_gib, 1), 'disk_free_gib': round(disk_gib, 1)}, sort_keys=True))


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive').resolve()


In [ ]:
import json, unicodedata
from pathlib import Path, PurePosixPath
raw_folder = INPUT_FOLDER.strip()
if not raw_folder:
    raw_folder = input('MyDrive-relative input folder: ').strip()
assert raw_folder and '\\' not in raw_folder, 'Use a MyDrive-relative POSIX path'
assert not any(unicodedata.category(ch) == 'Cc' for ch in raw_folder)
folder = PurePosixPath(raw_folder)
assert not folder.is_absolute() and folder.parts
assert all(part not in {'', '.', '..'} for part in folder.parts)
assert not raw_folder.endswith((
    '_legacy_control_5k_result', '_training_ablation',
    '_floor_recovery_diagnostic', '_learned_test_result',
    '_learned_test_diagnostics', '_learned_test_cache',
))
INPUT_PATH = DRIVE_ROOT.joinpath(*folder.parts).resolve()
INPUT_PATH.relative_to(DRIVE_ROOT)
assert INPUT_PATH.is_dir(), f'Input folder does not exist: {INPUT_PATH}'
CACHE_PATH = INPUT_PATH.with_name(INPUT_PATH.name + '_learned_test_cache')
REFERENCE_PATH = INPUT_PATH.with_name(INPUT_PATH.name + '_training_ablation')
RESULT_PATH = INPUT_PATH.with_name(INPUT_PATH.name + '_legacy_control_5k_result')
assert (REFERENCE_PATH / '_SUCCESS.json').is_file(), (
    'Completed A100 training diagnostic is required: ' + str(REFERENCE_PATH)
)
RUN_SPEC = {'schema_version': 1, 'input_folder': folder.as_posix(), 'runtime_profile': RUNTIME_PROFILE, 'publish': {'replace_owned_result': True}}
SPEC_PATH = Path('/content/legacy_control_export_spec.json')
with SPEC_PATH.open('w', encoding='utf-8') as handle:
    json.dump(RUN_SPEC, handle, sort_keys=True, separators=(',', ':'))
print(f'Input: {INPUT_PATH}')
print(f'Verified cache: {CACHE_PATH}')
print(f'A100 legacy reference: {REFERENCE_PATH}')
print(f'Dual PLY result: {RESULT_PATH}')


In [ ]:
import shutil, subprocess
from pathlib import Path
SOURCE_ROOT = Path('/content/gaussian-splatter-src')
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
REPOSITORY_URL = 'https://github.com/mehmettahacumurcu/gaussian-splatter.git'
COMMIT_SHA = '048f85f3b9544b57727b8756aea3d4815b899e99'
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(SOURCE_ROOT)], check=True)
subprocess.run(['git', '-C', str(SOURCE_ROOT), 'checkout', '--detach', COMMIT_SHA], check=True)
actual = subprocess.run(['git', '-C', str(SOURCE_ROOT), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
assert actual == COMMIT_SHA, 'Immutable source checkout mismatch'


In [ ]:
import subprocess
subprocess.run(['bash', 'colab/static_notebook_bootstrap.sh'], cwd=SOURCE_ROOT, check=True)


In [ ]:
import sys, time
sys.path.insert(0, str(SOURCE_ROOT))
from huggingface_hub import snapshot_download
from experiments.learned_quality.dependencies import (
    install_learned_environment, materialize_pinned_assets,
)
def download_with_backoff(**kwargs):
    kwargs['max_workers'] = 1
    for attempt in range(6):
        try:
            return snapshot_download(**kwargs)
        except Exception as error:
            if attempt == 5 or '429' not in str(error):
                raise
            delay = 240 + 60 * attempt
            print(f'Hugging Face rate limited; retrying in {delay}s...')
            time.sleep(delay)
def resolved_revision(local_path):
    metadata_root = Path(local_path) / '.cache' / 'huggingface' / 'download'
    revisions = set()
    for metadata in metadata_root.rglob('*.metadata'):
        first = metadata.read_text(encoding='utf-8').splitlines()[0].strip()
        if len(first) == 40:
            revisions.add(first)
    assert len(revisions) == 1, f'Cannot authenticate checkpoint: {local_path}'
    return revisions.pop()
LEARNED_ENV = install_learned_environment(Path(sys.executable).resolve())
LEARNED_ASSETS = materialize_pinned_assets(
    LEARNED_ENV, downloader=download_with_backoff, resolve_revision=resolved_revision,
)


In [ ]:
from experiments.learned_quality.dependencies import verify_learned_environment
MODEL_MANIFEST = verify_learned_environment(LEARNED_ENV, LEARNED_ASSETS)
assert MODEL_MANIFEST.path == Path('/content/learned-env/model_manifest.json')
print(f'Verified model manifest: {MODEL_MANIFEST.path}')


In [ ]:
import json, os, subprocess, traceback
from pathlib import Path
from google.colab import drive, runtime
environment = dict(os.environ)
environment['LEARNED_MODEL_MANIFEST'] = str(MODEL_MANIFEST.path)
environment['PYTHONUNBUFFERED'] = '1'
failure = None
try:
    completed = subprocess.run(
        [
            "/content/learned-env/bin/python",
            "-u", "-m",
            "scripts.learned_quality_legacy_control_export_run",
            "--spec", "/content/legacy_control_export_spec.json",
            "--source-revision", COMMIT_SHA,
            "--model-manifest", str(MODEL_MANIFEST.path),
        ], cwd=SOURCE_ROOT, env=environment, check=False,
    )
    receipt_path = Path('/content/legacy_control_export_result.json')
    if not receipt_path.is_file():
        raise RuntimeError('Legacy-control export ended without a receipt')
    receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
    if completed.returncode != 0 or receipt.get('status') != 'success':
        raise RuntimeError(f'Legacy-control export failed: {receipt}')
    actual = Path(receipt['final_path']).resolve()
    if actual != RESULT_PATH.resolve():
        raise RuntimeError(f'Unexpected dual PLY result path: {actual}')
    success = json.loads((actual / '_SUCCESS.json').read_text(encoding='utf-8'))
    if success.get('run_id') != receipt.get('run_id'):
        raise RuntimeError('Result marker does not match this run')
    required = (
        'raw_legacy_control_5k.ply',
        'polished_legacy_control_5k.ply',
        'polish_report.json', 'receipt.json', 'metrics.jsonl',
        'run_manifest.json', 'provenance.json',
    )
    missing = [name for name in required if not (actual / name).is_file()]
    if missing:
        raise RuntimeError(f'Published dual PLY result is incomplete: {missing}')
    raw_ply = actual / 'raw_legacy_control_5k.ply'
    polished_ply = actual / 'polished_legacy_control_5k.ply'
    print(f'Raw trainer PLY: {raw_ply}')
    print(f'Polished candidate PLY: {polished_ply}')
    print(f'Polish accepted: {receipt.get("polish_accepted")}')
    print(f'Polish reasons: {receipt.get("polish_reasons", [])}')
    print('No full 120K training was started.')
except BaseException as exc:
    failure = exc
    print(f'Run ended with {type(exc).__name__}: {exc}')
    traceback.print_exception(type(exc), exc, exc.__traceback__)
finally:
    print('Flushing outstanding Google Drive writes...')
    try:
        drive.flush_and_unmount()
    except BaseException as flush_error:
        print(f'Drive flush/unmount failed: {flush_error}')
    print('Releasing the Colab runtime now.')
    try:
        runtime.unassign()
    except BaseException as release_error:
        print(f'Runtime release request failed: {release_error}')
        if failure is None:
            failure = release_error
if failure is not None:
    raise failure
